# Fine-tune Cross-Encoder CV-JD v0.7

Fine-tune `cross-encoder/ms-marco-MiniLM-L-12-v2` với **MSE regression loss** trên dataset v0.6 — batch sinh mới hoàn toàn bằng `scripts/generate_synthetic_data.py` sau khi sửa generator (water-filling label balancing + skill-overlap authoring theo từng cặp + decision-priority rule cho poor_match).

| | |
|---|---|
| **Dataset** | v0.6 — 2,520 train / 540 validation / 540 test (3,600 pairs, cân bằng tuyệt đối 20%/class) |
| **Base model** | `cross-encoder/ms-marco-MiniLM-L-12-v2` |
| **Loss** | `MSELoss` regression (label = score / 100) |
| **Evaluator** | Spearman correlation + LabelAcc |
| **max_length** | 512 tokens |
| **Branch** | `experiment/cross-encoder-v0.7` |

**Mục tiêu**: xác nhận baseline trên dataset v0.6 — dataset nhỏ hơn v0.4 (7k) và v0.5 (13.35k) đáng kể, nhưng lần đầu tiên có class balance tuyệt đối (không lệch qua weak_match/excellent_match như các batch trước) và có poor_match thật (không phải 0% như v0.4/v0.5). So sánh với ceiling cũ:
- v0.2: 60.76% (7k, MSELoss)
- v0.5/v0.6 (data v0.5): 60.95% (13.35k, MSELoss + Spearman)

Nếu run 1 tiệm cận hoặc vượt ~60%, đó là tín hiệu tốt để tiếp tục sinh thêm data (hướng tới 7k+) cho lần chạy chính thức.


In [1]:
import os
if not os.path.exists('/content/Ai-Recruiter-Mini-Ai-Service'):
    !git clone https://github.com/DangHuuLong/Ai-Recruiter-Mini-Ai-Service /content/Ai-Recruiter-Mini-Ai-Service

%cd /content/Ai-Recruiter-Mini-Ai-Service
!git checkout experiment/cross-encoder-v0.7
!git pull origin experiment/cross-encoder-v0.7


/content/Ai-Recruiter-Mini-Ai-Service
Already on 'experiment/cross-encoder-v0.7'
Your branch is up to date with 'origin/experiment/cross-encoder-v0.7'.
From https://github.com/DangHuuLong/Ai-Recruiter-Mini-Ai-Service
 * branch            experiment/cross-encoder-v0.7 -> FETCH_HEAD
Already up to date.


In [2]:
!pip install -r requirements.txt


In [3]:
from pathlib import Path
import json

data_dir = Path("datasets/versions/v0.6/cross_encoder")
for split in ("train", "validation", "test"):
    path = data_dir / f"cross_encoder_{split}.jsonl"
    lines = path.read_text(encoding="utf-8").strip().splitlines()
    first = json.loads(lines[0])
    print(f"{split:<12}: {len(lines):>5} pairs  | keys: {list(first.keys())}")


train       :  2520 pairs  | keys: ['pair_id', 'cv_text', 'jd_text', 'score', 'label', 'true_label']
validation  :   540 pairs  | keys: ['pair_id', 'cv_text', 'jd_text', 'score', 'label', 'true_label']
test        :   540 pairs  | keys: ['pair_id', 'cv_text', 'jd_text', 'score', 'label', 'true_label']


In [4]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


CUDA available: True
Device: Tesla T4


## Debug run — sanity check (1 epoch, 40 samples)


In [5]:
!WANDB_MODE=disabled python -m training.fine_tune_cross_encoder \
    --data-dir datasets/versions/v0.6/cross_encoder \
    --loss mse \
    --evaluator spearman \
    --base-model cross-encoder/ms-marco-MiniLM-L-12-v2 \
    --epochs 1 \
    --batch-size 4 \
    --max-train-samples 40 \
    --max-eval-samples 20


2026-07-09 08:31:19.606344: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Base model : cross-encoder/ms-marco-MiniLM-L-12-v2
Data dir   : datasets/versions/v0.6/cross_encoder
Output dir : artifacts/models/cross-encoder-cv-jd-v0.1
Loss       : mse  (MSELoss)
Evaluator  : spearman
Epochs     : 1  |  Batch size: 4  |  Max length: 512

Train: 40 pairs  |  Val: 20 pairs

config.json: 100% 791/791 [00:00<00:00, 5.49MB/s]
model.safetensors: 100% 133M/133M [00:03<00:00, 37.4MB/s]
tokenizer_config.json: 1.33kB [00:00, 6.25MB/s]
vocab.txt: 232kB [00:00, 57.1MB/s]
tokenizer.json: 711kB [00:00, 134MB/s]
special_tokens_map.json: 100% 132/132 [00:00<00:00, 1.09MB/s]
Steps/epoch: 10  |  Warmup steps: 1

/usr/local/lib/python3.12/dist-packages/sentence_transfo

## Full training — 10 epochs, save to Google Drive


In [6]:
from google.colab import drive
drive.mount('/content/drive')

drive_base = "/content/drive/MyDrive/ai-recruiter"

import os, time
os.makedirs(f"{drive_base}/models/cross-encoder-cv-jd-v0.7", exist_ok=True)
time.sleep(3)

!WANDB_MODE=disabled python -m training.fine_tune_cross_encoder \
    --data-dir datasets/versions/v0.6/cross_encoder \
    --loss mse \
    --evaluator spearman \
    --base-model cross-encoder/ms-marco-MiniLM-L-12-v2 \
    --output-dir {drive_base}/models/cross-encoder-cv-jd-v0.7 \
    --report-path artifacts/reports/fine_tune_cross_encoder_v0.7_report.json \
    --epochs 10 \
    --batch-size 16


Mounted at /content/drive
2026-07-09 08:32:43.038550: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Base model : cross-encoder/ms-marco-MiniLM-L-12-v2
Data dir   : datasets/versions/v0.6/cross_encoder
Output dir : /content/drive/MyDrive/ai-recruiter/models/cross-encoder-cv-jd-v0.7
Loss       : mse  (MSELoss)
Evaluator  : spearman
Epochs     : 10  |  Batch size: 16  |  Max length: 512

Train: 2520 pairs  |  Val: 540 pairs

Steps/epoch: 158  |  Warmup steps: 15

/usr/local/lib/python3.12/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:234: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
  epoch  1/10  step   158 

In [7]:
import json
from pathlib import Path

report = json.loads(
    Path("artifacts/reports/fine_tune_cross_encoder_v0.7_report.json").read_text(encoding="utf-8")
)
print(f"Base model : {report['base_model']}")
print(f"Loss       : {report['loss']}")
print()
print(json.dumps(report["metrics"], indent=2))


Base model : cross-encoder/ms-marco-MiniLM-L-12-v2
Loss       : mse

{
  "validation": {
    "mae": 3.5381,
    "rmse": 5.9295,
    "label_accuracy": 0.9296,
    "pair_count": 540,
    "mean_predicted_score": 62.6558,
    "mean_target_score": 63.3296
  },
  "test": {
    "mae": 3.7039,
    "rmse": 5.7283,
    "label_accuracy": 0.913,
    "pair_count": 540,
    "mean_predicted_score": 64.5047,
    "mean_target_score": 64.637
  }
}


In [8]:
import shutil
from pathlib import Path

reports_dir = Path(drive_base) / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)
shutil.copy(
    "artifacts/reports/fine_tune_cross_encoder_v0.7_report.json",
    reports_dir / "fine_tune_cross_encoder_v0.7_report.json",
)
print(f"Saved to {reports_dir}")


Saved to /content/drive/MyDrive/ai-recruiter/reports
